# G-Code XGBoost v2: Shape-Invariant Features

**Problem with v1:** All features were absolute values (total moves, coordinate ranges, feed rates)
that encoded part **size** and **slicer settings** rather than **shape**. The model learned
"small parts sliced with Cura = gun" instead of "barrel-shaped = gun."

**This version:** Every feature is a **ratio**, **distribution**, or **normalised quantity**
that describes geometry regardless of scale, slicer, or printer.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'xgboost', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn',
    'datasets', 'tqdm', 'shap'])


In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import xgboost as xgb
from collections import defaultdict
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 1. Configuration


In [ ]:
HF_DATASET = "jungter/gcode-to-model-gn"
VAL_SPLIT = 0.2
RANDOM_SEED = 42

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 4,
    "learning_rate": 0.05,
    "n_estimators": 300,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "min_child_weight": 5,
    "reg_alpha": 0.5,
    "reg_lambda": 2.0,
    "gamma": 0.1,
    "random_state": RANDOM_SEED,
    "verbosity": 0,
}


## 2. Load Dataset


In [ ]:
from datasets import load_dataset

hf_ds = load_dataset(HF_DATASET, split="train")
print(f"Loaded {len(hf_ds)} samples from {HF_DATASET}")

df = pd.DataFrame({
    "gcode_text": hf_ds["gcode_text"],
    "label": hf_ds["label"],
    "category": hf_ds["category"],
    "part": hf_ds["part"],
    "filename": hf_ds["filename"],
})

print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"Unique parts: {df['part'].nunique()}")


## 3. Shape-Invariant Feature Extraction

Every feature is a **ratio**, **distribution statistic**, or **normalised value**
so that it describes geometry, not size or slicer settings.

| Category | What it captures | Example |
|----------|-----------------|--------|
| Aspect ratios | Proportions of the bounding box | Long narrow barrel vs. flat plate |
| Layer complexity | How movement density varies layer-to-layer | Uniform (cylinder) vs. varying (organic) |
| Movement directions | Distribution of X/Y/Z movement angles | Circular cross-sections vs. angular |
| Normalised density | Moves per unit volume | Solid vs. hollow |


In [ ]:
import math

def extract_shape_features(gcode_text: str) -> dict:
    """Extract shape-invariant features from g-code.
    
    All features are ratios, distributions, or normalised quantities
    that do NOT change with part scale, slicer settings, or printer config.
    """
    lines = gcode_text.splitlines()

    # ── Parse movements per layer ──
    current_layer = -1
    layer_moves = defaultdict(int)  # moves per layer
    layer_x = defaultdict(list)     # x positions per layer
    layer_y = defaultdict(list)     # y positions per layer

    g0_count = 0
    g1_count = 0
    x_vals, y_vals, z_vals = [], [], []

    # Movement direction tracking
    prev_x, prev_y, prev_z = None, None, None
    dx_list, dy_list, dz_list = [], [], []
    segment_lengths = []
    angles_xy = []  # direction angles in XY plane

    retraction_count = 0
    total_e_commands = 0

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.startswith(";"):
            if ";LAYER:" in stripped.upper():
                current_layer += 1
            continue

        is_g0 = stripped.startswith('G0 ') or stripped.startswith('G0\t')
        is_g1 = stripped.startswith('G1 ') or stripped.startswith('G1\t')

        if is_g0:
            g0_count += 1
        elif is_g1:
            g1_count += 1

        if not (is_g0 or is_g1):
            continue

        if current_layer >= 0:
            layer_moves[current_layer] += 1

        # Parse coordinates
        parts = stripped.split(";")[0].split()
        cur_x, cur_y, cur_z, cur_e = prev_x, prev_y, prev_z, None
        for p in parts:
            try:
                val_str = p[1:].strip('[]{}()')
                if p.startswith('X'):
                    cur_x = float(val_str)
                    x_vals.append(cur_x)
                    if current_layer >= 0:
                        layer_x[current_layer].append(cur_x)
                elif p.startswith('Y'):
                    cur_y = float(val_str)
                    y_vals.append(cur_y)
                    if current_layer >= 0:
                        layer_y[current_layer].append(cur_y)
                elif p.startswith('Z'):
                    cur_z = float(val_str)
                    z_vals.append(cur_z)
                elif p.startswith('E'):
                    cur_e = float(val_str)
                    total_e_commands += 1
                    if cur_e < 0:
                        retraction_count += 1
            except ValueError:
                pass

        # Track movement vectors
        if prev_x is not None and cur_x is not None and prev_y is not None and cur_y is not None:
            dx = cur_x - prev_x
            dy = cur_y - prev_y
            dz = (cur_z - prev_z) if (prev_z is not None and cur_z is not None) else 0.0

            seg_len = math.sqrt(dx*dx + dy*dy + dz*dz)
            if seg_len > 0.01:  # ignore tiny moves
                dx_list.append(abs(dx))
                dy_list.append(abs(dy))
                dz_list.append(abs(dz))
                segment_lengths.append(seg_len)

                # XY direction angle (for circularity detection)
                xy_len = math.sqrt(dx*dx + dy*dy)
                if xy_len > 0.01:
                    angle = math.atan2(dy, dx)  # -pi to pi
                    angles_xy.append(angle)

        if cur_x is not None: prev_x = cur_x
        if cur_y is not None: prev_y = cur_y
        if cur_z is not None: prev_z = cur_z

    # ── Compute features ──
    total_moves = g0_count + g1_count
    n_layers = max(current_layer + 1, 1)

    # Bounding box
    x_range = (max(x_vals) - min(x_vals)) if x_vals else 0.001
    y_range = (max(y_vals) - min(y_vals)) if y_vals else 0.001
    z_range = (max(z_vals) - min(z_vals)) if z_vals else 0.001

    # 1. ASPECT RATIOS (scale-invariant shape descriptors)
    aspect_xy = x_range / max(y_range, 0.001)
    aspect_xz = x_range / max(z_range, 0.001)
    aspect_yz = y_range / max(z_range, 0.001)

    # Elongation: max_dim / min_dim (1.0 = cube, high = long thin)
    dims = sorted([x_range, y_range, z_range])
    elongation = dims[2] / max(dims[0], 0.001)
    flatness = dims[0] / max(dims[1], 0.001)  # how flat (thin in one dim)

    # 2. MOVEMENT RATIOS
    extrusion_ratio = g1_count / max(total_moves, 1)
    retraction_ratio = retraction_count / max(total_e_commands, 1)

    # 3. LAYER COMPLEXITY DISTRIBUTION
    if layer_moves:
        moves_arr = np.array(list(layer_moves.values()), dtype=float)
        layer_complexity_mean = moves_arr.mean()
        layer_complexity_cv = moves_arr.std() / max(moves_arr.mean(), 1)  # coefficient of variation
        # How uniform are layers? Low CV = cylinder, High CV = organic/complex
        layer_complexity_skew = float(pd.Series(moves_arr).skew()) if len(moves_arr) > 2 else 0.0
        # Top/bottom layer ratio (does the shape taper?)
        n_quarter = max(len(moves_arr) // 4, 1)
        bottom_avg = moves_arr[:n_quarter].mean()
        top_avg = moves_arr[-n_quarter:].mean()
        taper_ratio = top_avg / max(bottom_avg, 1)
    else:
        layer_complexity_cv = 0.0
        layer_complexity_skew = 0.0
        taper_ratio = 1.0

    # 4. PER-LAYER FOOTPRINT CONSISTENCY
    layer_x_ranges = []
    layer_y_ranges = []
    for lyr in sorted(layer_x.keys()):
        if layer_x[lyr] and layer_y[lyr]:
            lxr = max(layer_x[lyr]) - min(layer_x[lyr])
            lyr_r = max(layer_y[lyr]) - min(layer_y[lyr])
            layer_x_ranges.append(lxr)
            layer_y_ranges.append(lyr_r)

    if layer_x_ranges:
        lxr_arr = np.array(layer_x_ranges)
        lyr_arr = np.array(layer_y_ranges)
        # Cross-section consistency: how much does the XY footprint change?
        footprint_x_cv = lxr_arr.std() / max(lxr_arr.mean(), 0.001)
        footprint_y_cv = lyr_arr.std() / max(lyr_arr.mean(), 0.001)
        # Per-layer aspect ratio consistency
        layer_aspects = lxr_arr / np.maximum(lyr_arr, 0.001)
        layer_aspect_cv = layer_aspects.std() / max(layer_aspects.mean(), 0.001)
        # Per-layer squareness (how circular/square is the cross-section?)
        layer_squareness = np.minimum(lxr_arr, lyr_arr) / np.maximum(np.maximum(lxr_arr, lyr_arr), 0.001)
        mean_squareness = layer_squareness.mean()
    else:
        footprint_x_cv = 0.0
        footprint_y_cv = 0.0
        layer_aspect_cv = 0.0
        mean_squareness = 0.0

    # 5. MOVEMENT DIRECTION DISTRIBUTION
    if dx_list:
        dx_arr = np.array(dx_list)
        dy_arr = np.array(dy_list)
        dz_arr = np.array(dz_list)
        total_displacement = dx_arr.sum() + dy_arr.sum() + dz_arr.sum()
        # What fraction of movement is in each axis?
        dx_frac = dx_arr.sum() / max(total_displacement, 0.001)
        dy_frac = dy_arr.sum() / max(total_displacement, 0.001)
        dz_frac = dz_arr.sum() / max(total_displacement, 0.001)
        # XY isotropy: are movements equally spread in X and Y? (1.0 = circular)
        xy_isotropy = min(dx_arr.sum(), dy_arr.sum()) / max(max(dx_arr.sum(), dy_arr.sum()), 0.001)
    else:
        dx_frac = 0.33
        dy_frac = 0.33
        dz_frac = 0.33
        xy_isotropy = 1.0

    # 6. ANGULAR DISTRIBUTION (circularity detector)
    if len(angles_xy) > 10:
        angle_arr = np.array(angles_xy)
        # Bin angles into 8 octants and check uniformity
        hist, _ = np.histogram(angle_arr, bins=8, range=(-np.pi, np.pi))
        hist_norm = hist / max(hist.sum(), 1)
        # Entropy of angle distribution (high = many directions = circular)
        hist_nonzero = hist_norm[hist_norm > 0]
        angle_entropy = -np.sum(hist_nonzero * np.log2(hist_nonzero))
        max_entropy = np.log2(8)  # max possible for 8 bins
        angle_uniformity = angle_entropy / max_entropy  # 1.0 = perfectly uniform = circular
    else:
        angle_uniformity = 0.5

    # 7. SEGMENT LENGTH DISTRIBUTION
    if segment_lengths:
        seg_arr = np.array(segment_lengths)
        seg_cv = seg_arr.std() / max(seg_arr.mean(), 0.001)
        # Short vs long move ratio
        median_seg = np.median(seg_arr)
        short_move_ratio = (seg_arr < median_seg * 0.5).sum() / len(seg_arr)
    else:
        seg_cv = 0.0
        short_move_ratio = 0.5

    # 8. NORMALISED DENSITY (moves per unit bounding volume)
    bounding_vol = x_range * y_range * z_range
    normalised_density = total_moves / max(bounding_vol, 0.001)
    # Log-scale to handle huge range
    log_density = math.log1p(normalised_density)

    features = {
        # Aspect ratios (3)
        "aspect_xy": aspect_xy,
        "aspect_xz": aspect_xz,
        "aspect_yz": aspect_yz,
        # Shape descriptors (2)
        "elongation": elongation,
        "flatness": flatness,
        # Movement ratios (2)
        "extrusion_ratio": extrusion_ratio,
        "retraction_ratio": retraction_ratio,
        # Layer complexity (3)
        "layer_complexity_cv": layer_complexity_cv,
        "layer_complexity_skew": layer_complexity_skew,
        "taper_ratio": taper_ratio,
        # Cross-section consistency (4)
        "footprint_x_cv": footprint_x_cv,
        "footprint_y_cv": footprint_y_cv,
        "layer_aspect_cv": layer_aspect_cv,
        "mean_squareness": mean_squareness,
        # Movement direction (4)
        "dx_frac": dx_frac,
        "dy_frac": dy_frac,
        "dz_frac": dz_frac,
        "xy_isotropy": xy_isotropy,
        # Angular distribution (1)
        "angle_uniformity": angle_uniformity,
        # Segment lengths (2)
        "segment_length_cv": seg_cv,
        "short_move_ratio": short_move_ratio,
        # Normalised density (1)
        "log_density": log_density,
    }
    return features


# Test on first sample
test_features = extract_shape_features(df['gcode_text'].iloc[0])
print(f'Shape-invariant feature count: {len(test_features)}')
for k, v in test_features.items():
    print(f'  {k:25s}: {v:.4f}')

## 4. Build Feature Matrix


In [ ]:
print("Extracting shape-invariant features...")
all_features = []
valid_indices = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        feats = extract_shape_features(row['gcode_text'])
        all_features.append(feats)
        valid_indices.append(idx)
    except Exception as e:
        print(f"Error processing {row['filename']}: {e}")

df = df.loc[valid_indices].reset_index(drop=True)
feature_df = pd.DataFrame(all_features)
print(f'\nExtracted {len(feature_df.columns)} features for {len(feature_df)} files')


## 5. Feature Leakage Check


In [ ]:
# Check if any single feature trivially separates the classes
print('Checking for feature leakage...')
leaking = []
for col in feature_df.columns:
    gun_vals = feature_df.loc[df['label']==1, col]
    nongun_vals = feature_df.loc[df['label']==0, col]
    gun_mean, gun_std = gun_vals.mean(), gun_vals.std()
    nongun_mean, nongun_std = nongun_vals.mean(), nongun_vals.std()
    separation = abs(gun_mean - nongun_mean) / max(gun_std + nongun_std, 1e-6)
    if separation > 2:
        print(f'  WARNING {col:25s}: gun={gun_mean:.3f}+/-{gun_std:.3f}, non-gun={nongun_mean:.3f}+/-{nongun_std:.3f}, sep={separation:.1f}')
        leaking.append(col)
    elif separation > 1:
        print(f'  watch   {col:25s}: gun={gun_mean:.3f}+/-{gun_std:.3f}, non-gun={nongun_mean:.3f}+/-{nongun_std:.3f}, sep={separation:.1f}')

if not leaking:
    print('  No features with suspiciously high class separation.')
else:
    print(f'\n  {len(leaking)} feature(s) may be leaking. Investigate before trusting results.')


## 6. Train/Val Split (Group-Aware)


In [ ]:
X = feature_df.values.astype(np.float32)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
y = df['label'].values
groups = df['part'].values

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SPLIT, random_state=RANDOM_SEED)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

train_parts = set(groups[train_idx])
val_parts = set(groups[val_idx])
overlap = train_parts & val_parts
print(f'Train: {len(y_train)} samples, {len(train_parts)} unique parts')
print(f'Val:   {len(y_val)} samples, {len(val_parts)} unique parts')
print(f'Part overlap: {len(overlap)} (should be 0)')
print(f'\nTrain labels: pos={y_train.sum()}, neg={len(y_train)-y_train.sum()}')
print(f'Val labels:   pos={y_val.sum()}, neg={len(y_val)-y_val.sum()}')


## 7. Cross-Validation


In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

cv_model = xgb.XGBClassifier(**XGB_PARAMS)
cv_scores = cross_val_score(
    cv_model, X_train, y_train, cv=sgkf, groups=groups[train_idx], scoring='roc_auc'
)

print('5-Fold Group-Stratified CV ROC-AUC:')
for i, s in enumerate(cv_scores):
    print(f'  Fold {i+1}: {s:.4f}')
print(f'  Mean:  {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

if cv_scores.mean() > 0.95:
    print('\nCV > 0.95 — still possibly too good. Check SHAP for remaining leakage.')
elif cv_scores.mean() > 0.80:
    print('\nCV in 0.80-0.95 range — looks reasonable!')
else:
    print('\nCV below 0.80 — features may not be discriminative enough.')


## 8. Train Final Model


In [ ]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / max(n_pos, 1)

final_params = {**XGB_PARAMS, 'scale_pos_weight': scale_pos_weight}
model = xgb.XGBClassifier(**final_params)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=10,
)

results = model.evals_result()
print(f"\nFinal train logloss: {results['validation_0']['logloss'][-1]:.4f}")
print(f"Final val logloss:   {results['validation_1']['logloss'][-1]:.4f}")


## 9. Evaluation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(len(results['validation_0']['logloss']))
axes[0].plot(epochs, results['validation_0']['logloss'], label='Train')
axes[0].plot(epochs, results['validation_1']['logloss'], label='Val')
axes[0].set_xlabel('Boosting Round')
axes[0].set_ylabel('Log Loss')
axes[0].set_title('Training Curves')
axes[0].legend()

train_ll = np.array(results['validation_0']['logloss'])
val_ll = np.array(results['validation_1']['logloss'])
axes[1].plot(epochs, val_ll - train_ll, color='red')
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1].set_xlabel('Boosting Round')
axes[1].set_ylabel('Val - Train Loss')
axes[1].set_title('Overfitting Gap')

plt.tight_layout()
plt.savefig('xgb_v2_training.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
y_pred = model.predict(X_val)
y_probs = model.predict_proba(X_val)[:, 1]

print('Classification Report:')
print(classification_report(y_val, y_pred, target_names=['non-gun', 'gun']))

if len(np.unique(y_val)) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['non-gun', 'gun'], yticklabels=['non-gun', 'gun'], ax=axes[0])
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_title('Confusion Matrix')

    gun_probs = y_probs[y_val == 1]
    nongun_probs = y_probs[y_val == 0]
    axes[1].hist(gun_probs, bins=30, alpha=0.6, label='Gun', color='red')
    axes[1].hist(nongun_probs, bins=30, alpha=0.6, label='Non-gun', color='blue')
    axes[1].set_xlabel('P(gun)')
    axes[1].set_title('Confidence Distribution')
    axes[1].legend()
    axes[1].axvline(x=0.5, color='k', linestyle='--', alpha=0.3)

    fraction_pos, mean_predicted = calibration_curve(y_val, y_probs, n_bins=10, strategy='uniform')
    axes[2].plot(mean_predicted, fraction_pos, 's-', label='Model', color='blue')
    axes[2].plot([0, 1], [0, 1], 'k--', label='Perfect')
    axes[2].set_xlabel('Predicted probability')
    axes[2].set_ylabel('Fraction positive')
    axes[2].set_title('Calibration')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig('xgb_v2_evaluation.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"ROC-AUC: {roc_auc_score(y_val, y_probs):.4f}")
    tn, fp, fn, tp = cm.ravel()
    print(f'TP={tp}, TN={tn}, FP={fp}, FN={fn}')

    high_conf_wrong = ((y_probs > 0.9) & (y_val == 0)).sum() + ((y_probs < 0.1) & (y_val == 1)).sum()
    print(f'\nOverconfident & wrong: {high_conf_wrong}')


## 10. Feature Importance & SHAP


In [ ]:
feature_names = list(feature_df.columns)
importances = model.feature_importances_
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(sorted_idx)), importances[sorted_idx])
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel('Importance')
ax.set_title('XGBoost v2 Feature Importance')
plt.tight_layout()
plt.savefig('xgb_v2_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features:')
for i in sorted_idx[-10:][::-1]:
    print(f'  {feature_names[i]:25s}: {importances[i]:.4f}')


In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_val)

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_val, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig('xgb_v2_shap.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Inference


In [ ]:
def classify_gcode(gcode_input: str, model, feature_columns, threshold=0.5):
    """Classify g-code. Accepts raw text or a filepath."""
    if os.path.isfile(gcode_input):
        with open(gcode_input, 'r', errors='ignore') as f:
            gcode_text = f.read()
    else:
        gcode_text = gcode_input

    features = extract_shape_features(gcode_text)
    feat_vec = np.array([[features[c] for c in feature_columns]], dtype=np.float32)
    feat_vec = np.nan_to_num(feat_vec, nan=0.0, posinf=0.0, neginf=0.0)

    prob = model.predict_proba(feat_vec)[0, 1]
    return prob >= threshold, float(prob)


feature_columns = list(feature_df.columns)
is_gun, confidence = classify_gcode('output.gcode', model, feature_columns)
print(f"File: output.gcode")
print(f"Prediction: {'GUN PART' if is_gun else 'NOT gun part'} (confidence: {confidence:.4f})")


## 12. Save Model


In [ ]:
SAVE_DIR = Path('./saved_model_xgb_v2')
SAVE_DIR.mkdir(exist_ok=True)

model.save_model(SAVE_DIR / 'xgb_classifier_v2.json')

config = {
    'model_type': 'xgboost_v2',
    'feature_columns': list(feature_df.columns),
    'n_features': len(feature_df.columns),
    'xgb_params': XGB_PARAMS,
    'description': 'Shape-invariant features only (no absolute sizes, no slicer config)',
}
with open(SAVE_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'Saved to {SAVE_DIR}')
print(f'Model size: {(SAVE_DIR / "xgb_classifier_v2.json").stat().st_size / 1024:.1f} KB')
